# 08 · Eval via vLLM (before / after DPO)

One notebook for **both** measurements. Change `RUN_NAME` + `MODEL` and rerun:
- **before:** `RUN_NAME="sft"`, vLLM serving the SFT model (`sft-toolcaller`)
- **after:**  `RUN_NAME="dpo"`, vLLM serving the DPO model

What it measures:
| set | size | question it answers |
|---|---|---|
| `test_v1` | 2,929 | is tool calling still good? (exact match, **over-abstain** rate) |
| `abstain_v1` | 300 | does it return `[]` when no tool fits? (**abstain rate**) |
| `general_irrelevant` | 10 | same, on hand-written obvious cases |

**Why vLLM and not the old HF eval?** Before and after must use the *same* generation code. vLLM greedy can differ slightly from HF greedy, so we re-measure SFT here instead of reusing 78.0.

**Needs:** vLLM running in WSL.

## 1 · Setup
Pick which run this is. Results go to `data/eval/results/<RUN_NAME>_vllm/`.

In [1]:
import sys, json, hashlib
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.prompts import build_messages
from src.evaluate import score_prediction, summarize, compare_runs
from src.abstention import build_abstention_records

RUN_NAME = "sft"              # "sft" now, "dpo" after training
MODEL    = "sft-toolcaller"   # the --served-model-name vLLM is using

EVAL = ROOT / "notebooks/data/eval"
OUT  = EVAL / "results" / f"{RUN_NAME}_vllm"
OUT.mkdir(parents=True, exist_ok=True)

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8")]

test = load_jsonl(EVAL / "test_v1.jsonl")
general_irr = [r for r in load_jsonl(EVAL / "general_prompts_v1.jsonl") if r["kind"] == "irrelevant_tools"]
print(len(test), "test |", len(general_irr), "general irrelevant")

2929 test | 10 general irrelevant


## 2 · Frozen abstain test set
Built from **`test_v1`** (never seen by SFT or DPO), half *removed* (hard) / half *swapped* (easy).
Created **once**, then only loaded, so before and after are scored on exactly the same 300 questions. The sha256 proves it didn't change.

In [2]:
ABSTAIN_PATH = EVAL / "abstain_v1.jsonl"

if not ABSTAIN_PATH.exists():
    recs = build_abstention_records(test, n=300, seed=3407, id_prefix="abst_")
    with open(ABSTAIN_PATH, "w", encoding="utf-8") as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("created", ABSTAIN_PATH.name)

abstain = load_jsonl(ABSTAIN_PATH)
sha = hashlib.sha256(ABSTAIN_PATH.read_bytes()).hexdigest()[:12]
print(len(abstain), "abstain questions | sha256", sha, "|", pd.Series([r["neg_kind"] for r in abstain]).value_counts().to_dict())

created abstain_v1.jsonl
300 abstain questions | sha256 4388990228e8 | {'removed': 150, 'swapped': 150}


## 3 · Greedy generation via vLLM
`temperature=0` = greedy = the model's single best answer (what users get). Parallel + resumable, same idea as in 07.

In [3]:
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")
print("vLLM serving:", client.models.list().data[0].id)

def generate(record):
    r = client.chat.completions.create(
        model=MODEL, messages=build_messages(record), temperature=0, max_tokens=512,
    )
    return r.choices[0].message.content

def run_set(records, name, workers=16):
    path = OUT / f"{name}.jsonl"
    done = {json.loads(l)["id"] for l in open(path, encoding="utf-8")} if path.exists() else set()
    todo = [r for r in records if r["id"] not in done]
    print(f"{name}: {len(done)} done, {len(todo)} to go")

    def work(r):
        try:
            return {"id": r["id"], "response": generate(r)}
        except Exception as e:
            return {"id": r["id"], "response": "", "error": str(e)[:200]}

    with open(path, "a", encoding="utf-8") as f, ThreadPoolExecutor(workers) as ex:
        for fut in tqdm(as_completed([ex.submit(work, r) for r in todo]), total=len(todo)):
            f.write(json.dumps(fut.result(), ensure_ascii=False) + "\n")
            f.flush()
    return {g["id"]: g for g in load_jsonl(path)}

gen_test    = run_set(test, "test_v1")
gen_abstain = run_set(abstain, "abstain_v1")
gen_general = run_set(general_irr, "general_irrelevant")

vLLM serving: sft-toolcaller
test_v1: 0 done, 2929 to go


  0%|          | 0/2929 [00:00<?, ?it/s]

abstain_v1: 0 done, 300 to go


  0%|          | 0/300 [00:00<?, ?it/s]

general_irrelevant: 0 done, 10 to go


  0%|          | 0/10 [00:00<?, ?it/s]

## 4 · Score
Same `score_prediction` as every other stage. For abstain sets the gold answer is `[]`, so **exact_match = abstained**.
**Over-abstain** = model returned `[]` on a normal test question (`no_call_when_needed`). This must stay near 0 after DPO.

In [4]:
def score_set(records, gens):
    rows = []
    for r in records:
        g = gens.get(r["id"], {})
        rows.append({"id": r["id"], "kind": r.get("neg_kind", r.get("kind", "")),
                     "error": "error" in g, **score_prediction(g.get("response", ""), r)})
    return pd.DataFrame(rows)

df_test    = score_set(test, gen_test)
df_abstain = score_set(abstain, gen_abstain)
df_general = score_set(general_irr, gen_general)
for name, d in [("test", df_test), ("abstain", df_abstain), ("general", df_general)]:
    print(f"{name}: {d.error.sum()} request errors")

df_test.to_csv(OUT / "scored_test.csv", index=False)
df_abstain.to_csv(OUT / "scored_abstain.csv", index=False)

print("\n=== test_v1 ===")
display(summarize(df_test, n_boot=1000))
print("failure mix (%):")
display((100 * df_test.failure_type.value_counts(normalize=True)).round(1))

test: 0 request errors
abstain: 0 request errors
general: 0 request errors

=== test_v1 ===


,value,ci_low,ci_high
metric,,,
exact_match,81.4,79.9,82.7
exact_match_lenient,81.4,80.0,82.7
function_match,97.3,96.7,97.8
argument_f1,91.8,91.0,92.5
parse_valid,99.8,99.6,99.9
schema_valid,99.5,99.2,99.7
invented_tool,0.0,0.0,0.0
invented_argument,0.2,0.1,0.4


failure mix (%):


failure_type
correct                 81.4
wrong_value             15.7
wrong_call_count         1.6
wrong_tool               0.9
parse_fail               0.2
unknown_arg              0.1
missing_required_arg     0.1
Name: proportion, dtype: float64

## 5 · Headline metrics + save
The numbers for the README table. `metrics.json` makes the after-DPO comparison a simple file diff.

In [5]:
metrics = {
    "run": RUN_NAME, "model": MODEL, "decoding": "greedy (vLLM)", "abstain_set_sha256": sha,
    "test_exact_match":   round(100 * df_test.exact_match.mean(), 1),
    "test_function_match": round(100 * df_test.function_match.mean(), 1),
    "test_argument_f1":   round(100 * df_test.argument_f1.mean(), 1),
    "test_over_abstain":  round(100 * (df_test.failure_type == "no_call_when_needed").mean(), 1),
    "abstain_rate":       round(100 * df_abstain.exact_match.mean(), 1),
    "abstain_rate_by_kind": (100 * df_abstain.groupby("kind").exact_match.mean()).round(1).to_dict(),
    "general_irrelevant_abstain": round(100 * df_general.exact_match.mean(), 1),
}
json.dump(metrics, open(OUT / "metrics.json", "w"), indent=2)
pd.Series(metrics)

run                                                        sft
model                                           sft-toolcaller
decoding                                         greedy (vLLM)
abstain_set_sha256                                4388990228e8
test_exact_match                                          81.4
test_function_match                                       97.3
test_argument_f1                                          91.8
test_over_abstain                                          0.0
abstain_rate                                               0.0
abstain_rate_by_kind          {'removed': 0.0, 'swapped': 0.0}
general_irrelevant_abstain                                 0.0
dtype: object

## 6 · Sanity check vs the old HF eval (SFT only)
vLLM and HF greedy should agree on almost every question. A big gap would mean the served model or prompt differs.

In [6]:
hf = pd.read_csv(EVAL / "results/sft_qwen2.5-0.5b/scored.csv")
if RUN_NAME == "sft":
    print(compare_runs(hf, df_test))   # a = HF, b = vLLM

{'n_paired': 2929, 'a': 78.0, 'b': 81.4, 'fixed': 142, 'broke': 43, 'both_correct': 2241, 'both_wrong': 503, 'mcnemar_p': 0.0}


## 7 · Compare before vs after (run when both exist)

In [7]:
before, after = EVAL / "results/sft_vllm", EVAL / "results/dpo_vllm"
if (after / "metrics.json").exists():
    mb, ma = json.load(open(before / "metrics.json")), json.load(open(after / "metrics.json"))
    keys = ["test_exact_match", "test_argument_f1", "test_over_abstain", "abstain_rate", "general_irrelevant_abstain"]
    display(pd.DataFrame({"SFT": [mb[k] for k in keys], "DPO": [ma[k] for k in keys]}, index=keys))
    print(compare_runs(pd.read_csv(before / "scored_test.csv"), pd.read_csv(after / "scored_test.csv")))
else:
    print("No DPO results yet: train, serve the DPO model, set RUN_NAME='dpo', rerun.")

No DPO results yet: train, serve the DPO model, set RUN_NAME='dpo', rerun.
